# Download `google/medasr` and upload to your public Hugging Face repo

This notebook downloads the model files from `google/medasr` and uploads them to a **public** repository under your Hugging Face account.

In [1]:
%pip install -q huggingface_hub

## 1) Authenticate

Set your Hugging Face token in the environment (`HF_TOKEN`) before running, or paste it in the prompt if needed.

Token scope needed: **Write** access.

In [2]:
import os
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN_WRITE")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print('Logged in with HF_TOKEN from environment.')
else:
    print('HF_TOKEN not found in environment. You will be prompted to paste a token.')
    login(add_to_git_credential=False)

Logged in with HF_TOKEN from environment.


## 2) Configure source and destination

Set `DEST_REPO_ID` to your target repo in the form `username/repo-name`.

In [3]:
SOURCE_REPO_ID = 'google/medasr'
DEST_REPO_ID = 'gabrielbuzzi/medasr-public'  # TODO: change this
LOCAL_DIR = './google_medasr_snapshot'

## 3) Download model snapshot from source repo

In [4]:
from huggingface_hub import snapshot_download

local_path = snapshot_download(
    repo_id=SOURCE_REPO_ID,
    local_dir=LOCAL_DIR,
    local_dir_use_symlinks=False
)

print(f'Download complete: {local_path}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Download complete: /content/google_medasr_snapshot


## 4) Create destination public repo (if needed) and upload

In [7]:
from huggingface_hub import HfApi

if DEST_REPO_ID.startswith('your-username/'):
    raise ValueError('Please set DEST_REPO_ID to your actual username/repo-name before running.')

api = HfApi()

api.create_repo(
    repo_id=DEST_REPO_ID,
    repo_type='model',
    private=False,
    exist_ok=True
)
print(f'Repo ready: https://huggingface.co/{DEST_REPO_ID}')

api.upload_folder(
    repo_id=DEST_REPO_ID,
    repo_type='model',
    folder_path=LOCAL_DIR,
    commit_message='Upload google/medasr snapshot',
    token=hf_token
)

print('Upload complete.')
print(f'Published model: https://huggingface.co/{DEST_REPO_ID}')

Repo ready: https://huggingface.co/gabrielbuzzi/medasr-public


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...r_snapshot/test_audio.wav: 100%|##########| 1.40MB / 1.40MB            

  ...asr_snapshot/spiece.model: 100%|##########|  247kB /  247kB            

  ...napshot/model.safetensors:   6%|5         | 25.1MB /  421MB            

  ...edasr_snapshot/lm_6.kenlm:   4%|3         | 25.1MB /  704MB            

Upload complete.
Published model: https://huggingface.co/gabrielbuzzi/medasr-public


## Notes

- If `google/medasr` is gated, your token/account must have access first.
- Large uploads can take time; rerunning `upload_folder` is safe and only uploads changed files.